# AIOps Log Analysis System

This is a comprehensive AIOps (Artificial Intelligence for IT Operations) system that automatically analyzes log data to detect anomalies, perform intelligent analysis, and generate remediation plans.

## Features
- **Simple Log Analysis**: Basic anomaly detection and analysis workflow
- **Advanced Log Analysis**: Multi-stage analysis with severity detection and remediation planning


## Requirements
- OpenAI API Key
- Python 3.8+
- LangGraph, LangChain, and related dependencies


In [2]:
# Essential Imports for Log Analysis Graphs
import os
import getpass
from typing import TypedDict, List, Dict, Any, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
import nest_asyncio

# Apply async support for Jupyter
nest_asyncio.apply()

print("✅ Essential imports loaded!")


✅ Essential imports loaded!


In [3]:
# API Key Setup
print("🔑 Setting up API keys...")
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
print("✅ API keys configured!")


🔑 Setting up API keys...
✅ API keys configured!


In [4]:
# State Classes for Log Analysis
class LogAnalysisState(TypedDict):
    """State class for AIOps log analysis workflows"""
    messages: Annotated[List[BaseMessage], add_messages]
    log_data: str
    anomaly_results: str
    analysis_results: str
    remediation_plan: str
    current_agent: Annotated[List[str], add_messages]

print("✅ State classes defined!")


✅ State classes defined!


In [5]:
# Helper Functions
def create_supervisor_helper(name: str, prompt: str):
    """Create supervisor helper function"""
    def supervisor_node(state: LogAnalysisState):
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
        messages = state["messages"]
        messages.append(SystemMessage(content=prompt))
        
        response = llm.invoke(messages)
        
        return {
            "messages": [response],
            "current_agent": [name]
        }
    return supervisor_node

def create_agent_node_helper(name: str, prompt: str, tools: List[Tool]):
    """Create agent node helper function"""
    def agent_node(state: LogAnalysisState):
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
        messages = state["messages"]
        messages.append(SystemMessage(content=prompt))
        
        response = llm.invoke(messages)
        
        return {
            "messages": [response],
            "current_agent": [name]
        }
    return agent_node

print("✅ Helper functions created!")


✅ Helper functions created!


In [6]:
# SIMPLE: Basic Log Analysis Graph
def create_simple_log_analysis_graph():
    """Create a simple log analysis graph without complex tool call sequences"""
    
    # Create simple tools (no complex tool calls)
    def simple_anomaly_detection(log_data: str) -> str:
        """Simple anomaly detection without tool calls"""
        if "ERROR" in log_data or "CRITICAL" in log_data:
            return f"🚨 ANOMALY DETECTED: {log_data[:50]}... - Critical error found!"
        return f"✅ NO ANOMALIES: {log_data[:50]}... - Log appears normal"

    def simple_log_analysis(log_data: str) -> str:
        """Simple log analysis without tool calls"""
        if "Database connection failed" in log_data:
            return f"🔍 ANALYSIS: Database connectivity issue detected. Check network and database server status."
        return f"📊 ANALYSIS: {log_data[:50]}... - Standard log entry analyzed"

    # Define simple agent functions
    def log_supervisor(state: LogAnalysisState):
        supervisor_prompt = """You are a log analysis supervisor. Coordinate the analysis process."""

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
        messages = state["messages"]
        messages.append(SystemMessage(content=supervisor_prompt))

        response = llm.invoke(messages)

        return {
            "messages": [response],
            "current_agent": ["log_supervisor"]
        }

    def anomaly_agent(state: LogAnalysisState):
        log_data = state.get("log_data", "")
        anomaly_result = simple_anomaly_detection(log_data)

        return {
            "messages": [AIMessage(content=f"Anomaly Detection Result: {anomaly_result}")],
            "anomaly_results": anomaly_result,
            "current_agent": ["anomaly_agent"]
        }

    def analysis_agent(state: LogAnalysisState):
        log_data = state.get("log_data", "")
        analysis_result = simple_log_analysis(log_data)

        return {
            "messages": [AIMessage(content=f"Analysis Result: {analysis_result}")],
            "analysis_results": analysis_result,
            "current_agent": ["analysis_agent"]
        }

    # Create graph
    graph = StateGraph(LogAnalysisState)

    # Add nodes
    graph.add_node("log_supervisor", log_supervisor)
    graph.add_node("anomaly_agent", anomaly_agent)
    graph.add_node("analysis_agent", analysis_agent)

    # Add simple edges
    graph.add_edge("log_supervisor", "anomaly_agent")
    graph.add_edge("anomaly_agent", "analysis_agent")
    graph.add_edge("analysis_agent", END)  # End the graph here instead of looping back

    graph.set_entry_point("log_supervisor")

    return graph.compile()

# Create the simple graph
simple_log_graph = create_simple_log_analysis_graph()
print("✅ Simple Log Analysis Graph created!")


✅ Simple Log Analysis Graph created!


In [7]:
# ADVANCED: Multi-Stage Log Analysis Graph with Decision Logic
def create_advanced_log_analysis_graph():
    """Create an advanced log analysis graph with decision logic and multiple stages"""
    
    # Advanced analysis functions
    def advanced_anomaly_detection(log_data: str) -> dict:
        """Advanced anomaly detection with severity levels"""
        severity = "LOW"
        anomaly_type = "NONE"

        if "CRITICAL" in log_data or "FATAL" in log_data:
            severity = "CRITICAL"
            anomaly_type = "SYSTEM_FAILURE"
        elif "ERROR" in log_data:
            severity = "HIGH"
            anomaly_type = "APPLICATION_ERROR"
        elif "WARNING" in log_data:
            severity = "MEDIUM"
            anomaly_type = "WARNING"
        elif "INFO" in log_data:
            severity = "LOW"
            anomaly_type = "INFORMATIONAL"

        return {
            "severity": severity,
            "type": anomaly_type,
            "detected": severity != "LOW",
            "message": f"🔍 {severity} {anomaly_type}: {log_data[:50]}..."
        }

    def advanced_log_analysis(log_data: str, anomaly_info: dict) -> dict:
        """Advanced log analysis with context-aware insights"""
        insights = []
        recommendations = []

        if "Database connection failed" in log_data:
            insights.append("Database connectivity issue detected")
            insights.append("Potential network or server problem")
            recommendations.append("Check database server status")
            recommendations.append("Verify network connectivity")
            recommendations.append("Review connection pool settings")
        elif "Memory" in log_data:
            insights.append("Memory-related issue detected")
            recommendations.append("Check memory usage and allocation")
        elif "CPU" in log_data:
            insights.append("CPU-related issue detected")
            recommendations.append("Monitor CPU usage and performance")

        return {
            "insights": insights,
            "recommendations": recommendations,
            "priority": "HIGH" if anomaly_info["severity"] == "CRITICAL" else "MEDIUM"
        }

    def remediation_planner(analysis_info: dict) -> dict:
        """Advanced remediation planning based on analysis"""
        if analysis_info["priority"] == "HIGH":
            return {
                "immediate_actions": [
                    "Alert on-call team",
                    "Check system status",
                    "Verify backup systems"
                ],
                "investigation_steps": [
                    "Review recent changes",
                    "Check system metrics",
                    "Analyze error patterns"
                ],
                "prevention_measures": [
                    "Implement monitoring",
                    "Set up alerts",
                    "Review procedures"
                ]
            }
        else:
            return {
                "immediate_actions": ["Log for review"],
                "investigation_steps": ["Schedule analysis"],
                "prevention_measures": ["Monitor trends"]
            }

    # Advanced agent functions
    def log_supervisor(state: LogAnalysisState):
        supervisor_prompt = """You are an advanced AIOps supervisor. Analyze the log data and coordinate
        a comprehensive incident response. Provide strategic guidance for the analysis team."""

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
        messages = state["messages"]
        messages.append(SystemMessage(content=supervisor_prompt))

        response = llm.invoke(messages)

        return {
            "messages": [response],
            "current_agent": ["log_supervisor"]
        }

    def anomaly_agent(state: LogAnalysisState):
        log_data = state.get("log_data", "")
        anomaly_result = advanced_anomaly_detection(log_data)

        return {
            "messages": [AIMessage(content=f"Advanced Anomaly Detection: {anomaly_result['message']}")],
            "anomaly_results": anomaly_result["message"],
            "current_agent": ["anomaly_agent"]
        }

    def analysis_agent(state: LogAnalysisState):
        log_data = state.get("log_data", "")
        anomaly_info = {"severity": "HIGH", "type": "APPLICATION_ERROR"}  # Simplified for demo
        analysis_result = advanced_log_analysis(log_data, anomaly_info)

        return {
            "messages": [AIMessage(content=f"Advanced Analysis: {analysis_result['insights']} | Recommendations: {analysis_result['recommendations']}")],
            "analysis_results": str(analysis_result),
            "current_agent": ["analysis_agent"]
        }

    def remediation_agent(state: LogAnalysisState):
        analysis_info = {"priority": "HIGH"}  # Simplified for demo
        remediation_result = remediation_planner(analysis_info)

        return {
            "messages": [AIMessage(content=f"Remediation Plan: {remediation_result['immediate_actions']}")],
            "remediation_plan": str(remediation_result),
            "current_agent": ["remediation_agent"]
        }

    # Create advanced graph
    graph = StateGraph(LogAnalysisState)

    # Add nodes
    graph.add_node("log_supervisor", log_supervisor)
    graph.add_node("anomaly_agent", anomaly_agent)
    graph.add_node("analysis_agent", analysis_agent)
    graph.add_node("remediation_agent", remediation_agent)

    # Add advanced edges with decision logic
    graph.add_edge("log_supervisor", "anomaly_agent")
    graph.add_edge("anomaly_agent", "analysis_agent")
    graph.add_edge("analysis_agent", "remediation_agent")
    graph.add_edge("remediation_agent", END)  # End after remediation

    graph.set_entry_point("log_supervisor")

    return graph.compile()

# Create the advanced graph
advanced_log_graph = create_advanced_log_analysis_graph()
print("✅ Advanced Log Analysis Graph created!")


✅ Advanced Log Analysis Graph created!


In [8]:
# Test Simple Log Analysis Graph
print("🧪 Testing Simple Log Analysis Graph")
print("=" * 50)

test_log_data = "ERROR: Database connection failed at 2024-01-15 10:30:45. Service unavailable. Retrying connection..."

try:
    # Test the simple graph
    result = simple_log_graph.invoke({
        "messages": [HumanMessage(content=test_log_data)],
        "log_data": test_log_data,
        "anomaly_results": "",
        "analysis_results": "",
        "remediation_plan": "",
        "current_agent": []
    })

    print("✅ Simple Log Analysis Graph Test Completed!")
    print(f"📊 Analysis Results: {result.get('analysis_results', 'No results')[:100]}...")
    print(f"🔍 Anomaly Results: {result.get('anomaly_results', 'No results')[:100]}...")
    print(f"📝 Current Agent: {result.get('current_agent', 'Unknown')}")

except Exception as e:
    print(f"❌ Simple Log Analysis Graph Test Failed: {e}")
    import traceback
    traceback.print_exc()


🧪 Testing Simple Log Analysis Graph
✅ Simple Log Analysis Graph Test Completed!
📊 Analysis Results: 🔍 ANALYSIS: Database connectivity issue detected. Check network and database server status....
🔍 Anomaly Results: 🚨 ANOMALY DETECTED: ERROR: Database connection failed at 2024-01-15 10... - Critical error found!...
📝 Current Agent: [HumanMessage(content='log_supervisor', additional_kwargs={}, response_metadata={}, id='d77df167-5b06-4536-9c80-685704fba2c4'), HumanMessage(content='anomaly_agent', additional_kwargs={}, response_metadata={}, id='f6dbac68-05ca-4c0e-af92-92ead319fcb7'), HumanMessage(content='analysis_agent', additional_kwargs={}, response_metadata={}, id='cc659a85-a6cb-4ce2-839e-a115a155ac47')]


In [9]:
# Test Advanced Log Analysis Graph with Streaming
print("🔄 Testing Advanced Log Analysis Graph with Streaming")
print("=" * 60)

test_log_data = "ERROR: Database connection failed at 2024-01-15 10:30:45. Service unavailable. Retrying connection..."

try:
    # Test with streaming
    print("🚀 Starting advanced streaming test...")
    for s in advanced_log_graph.stream(
        {
            "messages": [HumanMessage(content=test_log_data)],
            "log_data": test_log_data,
            "anomaly_results": "",
            "analysis_results": "",
            "remediation_plan": "",
            "current_agent": []
        },
        {"recursion_limit": 10}
    ):
        if "__end__" not in s:
            print(s)
            print("---")

    print("✅ Advanced streaming test completed!")

except Exception as e:
    print(f"❌ Advanced streaming test failed: {e}")
    import traceback
    traceback.print_exc()


🔄 Testing Advanced Log Analysis Graph with Streaming
🚀 Starting advanced streaming test...
{'log_supervisor': {'messages': [AIMessage(content='### Incident Response Plan for Database Connection Failure\n\n#### Incident Overview\n- **Timestamp**: 2024-01-15 10:30:45\n- **Error**: Database connection failed\n- **Status**: Service unavailable\n- **Action**: Retrying connection\n\n#### Immediate Actions\n1. **Acknowledge the Incident**: Notify all relevant stakeholders (IT team, database administrators, and management) about the incident.\n2. **Assess Impact**: Determine which services or applications are affected by the database connection failure. Identify critical business functions that may be impacted.\n3. **Establish Communication**: Set up a communication channel (e.g., Slack, email) for real-time updates and coordination among team members.\n\n#### Investigation Steps\n1. **Log Analysis**:\n   - Review database logs for any error messages or warnings leading up to the connection fa

In [10]:
advanced_log_analysis = create_advanced_log_analysis_graph()
print(advanced_log_analysis.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	log_supervisor(log_supervisor)
	anomaly_agent(anomaly_agent)
	analysis_agent(analysis_agent)
	remediation_agent(remediation_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> log_supervisor;
	analysis_agent --> remediation_agent;
	anomaly_agent --> analysis_agent;
	log_supervisor --> anomaly_agent;
	remediation_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

